# Vision Transformers — patch and attention inspection

The cells run a small, untrained NumPy forward path; no checkpoint or framework is required.

In [ ]:
from pathlib import Path
import importlib.util

lesson_rel = Path('phases/04-computer-vision/14-vision-transformers')
candidates = [
    Path.cwd() / lesson_rel / 'code' / 'main.py',
    Path.cwd() / 'code' / 'main.py',
    Path.cwd().parent / 'code' / 'main.py',
    Path.cwd().parent.parent / 'code' / 'main.py',
]
main_path = next((p.resolve() for p in candidates if p.is_file()), None)
if main_path is None or main_path.parent.parent.name != '14-vision-transformers':
    raise FileNotFoundError(f'lesson entrypoint not found; checked {candidates}')
spec = importlib.util.spec_from_file_location('cv04_l14_notebook', main_path)
lesson = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(lesson)
print(f'loaded {main_path}')

In [ ]:
import numpy as np
rng = np.random.default_rng(7)
images = rng.normal(size=(2, 3, 32, 32))
patches = lesson.patchify(images, 8)
print('patches:', patches.shape, 'raw width:', patches.shape[-1])

In [ ]:
result = lesson.vit_forward(images, patch_size=8, dim=24, num_heads=3, num_classes=4, seed=7)
assert result['tokens'].shape == (2, 17, 24)
assert result['attention'].shape == (2, 3, 17, 17)
np.testing.assert_allclose(result['attention'].sum(axis=-1), 1.0)
print('tokens:', result['tokens'].shape, 'logits:', result['logits'].shape, 'row sum:', result['attention'][0, 0, 0].sum())